# ADSL Pipeline — CDISCPILOT01

This notebook reproduces the [admiral ADSL vignette](https://pharmaverse.github.io/admiral/articles/adsl.html) programming flow using **Fleet + Polars**.

The ten major steps mirror admiral's derivation sequence:

| Step | Admiral function | Fleet equivalent |
|------|-----------------|---------------------|
| 1 | Read DM, AE, EX, DS | `pl.read_csv()` |
| 2 | Build SDTM domains | `df.sdtm` pipeline |
| 3 | Treatment variables | `.rename("TRT01P", ...)` |
| 4 | Treatment dates | `.first_date()` / `.last_date()` |
| 5 | Treatment duration | `.duration()` |
| 6 | Disposition | Polars joins from DS |
| 7 | Age grouping | `.condderivation()` |
| 8 | Population flags | `.eligibility_flag()` |
| 9 | Last date known alive | `pl.max_horizontal()` |
| 10 | AE counts | `.count_events()` |

## Setup

In [1]:
from __future__ import annotations
from pathlib import Path

import polars as pl
import fleet  # noqa: F401 — registers .sdtm / .adam / .adsl namespace extensions

# Paths — adjust if running from a different directory
REPO = Path("../../")   # notebook lives in docs/examples/
RAW = REPO / "datasets/cdiscpilot01/raw"
OUT = REPO / "outputs/cdiscpilot01"

print(f"Fleet {fleet.__version__}")
print(f"Polars    {pl.__version__}")

Fleet 0.1.0
Polars    1.39.0


## Step 1 — Read Raw Data

In [2]:
dm_raw = pl.read_csv(RAW / "dm_raw.csv")
ae_raw = pl.read_csv(RAW / "ae_raw.csv")

print(f"dm_raw : {dm_raw.shape[0]:>3} rows  \ncols: {dm_raw.columns}")
print(f"\nae_raw : {ae_raw.shape[0]:>3} rows  \ncols: {ae_raw.columns}")
dm_raw.head(3)

dm_raw :   5 rows  
cols: ['patient_number', 'site_id', 'age', 'sex', 'race', 'ethnic', 'arm_code', 'arm_description', 'first_dose_date', 'last_dose_date', 'informed_consent_date']

ae_raw :   5 rows  
cols: ['patient_number', 'ae_term', 'ae_start_date', 'ae_end_date', 'ae_severity', 'ae_serious', 'ae_outcome', 'ae_action_taken', 'ae_relationship']


patient_number,site_id,age,sex,race,ethnic,arm_code,arm_description,first_dose_date,last_dose_date,informed_consent_date
str,i64,i64,str,str,str,str,str,str,str,str
"""01-701-1015""",701,63,"""F""","""WHITE""","""NOT HISPANIC OR LATINO""","""Pbo""","""Placebo""","""2014-01-02""","""2014-07-02""","""2014-01-01"""
"""01-701-1023""",701,64,"""F""","""WHITE""","""NOT HISPANIC OR LATINO""","""Xan_Lo""","""Xanomeline Low Dose""","""2014-01-03""","""2014-07-04""","""2014-01-02"""
"""01-701-1028""",701,71,"""M""","""WHITE""","""NOT HISPANIC OR LATINO""","""Xan_Hi""","""Xanomeline High Dose""","""2014-01-04""","""2014-07-05""","""2014-01-03"""


## Step 2a — SDTM DM

Admiral starts from `dm` and selects/mutates. Fleet maps the raw columns through the `.sdtm` pipeline.

In [3]:
dm = (
    dm_raw.sdtm
    .hardcode_no_ct("STUDYID",  value="CDISCPILOT01")
    .hardcode_no_ct("DOMAIN",   value="DM")
    .assign_no_ct(  "USUBJID",  source="patient_number")
    .assign_no_ct(  "SUBJID",   source="patient_number")
    .derivation(    "SITEID",   expression="pl.col('site_id').cast(pl.String)")
    .assign_no_ct(  "AGE",      source="age")
    .hardcode_no_ct("AGEU",     value="YEARS")
    .assign_ct("SEX",   source="sex",    ct_codelist="C66731",
               ct_mapping={"F": "F", "M": "M"})
    .assign_ct("RACE",  source="race",   ct_codelist="C74457",
               ct_mapping={
                   "WHITE":                       "WHITE",
                   "BLACK OR AFRICAN AMERICAN":   "BLACK OR AFRICAN AMERICAN",
               })
    .assign_ct("ETHNIC", source="ethnic", ct_codelist="C66790",
               ct_mapping={
                   "HISPANIC OR LATINO":     "HISPANIC OR LATINO",
                   "NOT HISPANIC OR LATINO": "NOT HISPANIC OR LATINO",
               })
    .if_then_else("ARMCD",
                  when=[
                      {"condition": pl.col("arm_code") == "Pbo",    "value": "PBO"},
                      {"condition": pl.col("arm_code") == "Xan_Lo", "value": "XAN_LO"},
                      {"condition": pl.col("arm_code") == "Xan_Hi", "value": "XAN_HI"},
                  ],
                  else_value="SCRNFAIL")
    .assign_no_ct(  "ARM",      source="arm_description")
    .assign_no_ct(  "ACTARMCD", source="arm_code")
    .assign_no_ct(  "ACTARM",   source="arm_description")
    .assign_datetime("RFSTDTC", source="first_dose_date",       mode="date")
    .assign_datetime("RFENDTC", source="last_dose_date",        mode="date")
    .assign_datetime("RFICDTC", source="informed_consent_date", mode="date")
    .to_frame()
)

print(f"SDTM DM: {dm.shape[0]} subjects × {dm.shape[1]} variables")
dm.head(3)

SDTM DM: 5 subjects × 28 variables


patient_number,site_id,age,sex,race,ethnic,arm_code,arm_description,first_dose_date,last_dose_date,informed_consent_date,STUDYID,DOMAIN,USUBJID,SUBJID,SITEID,AGE,AGEU,SEX,RACE,ETHNIC,ARMCD,ARM,ACTARMCD,ACTARM,RFSTDTC,RFENDTC,RFICDTC
str,i64,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,str,str,str
"""01-701-1015""",701,63,"""F""","""WHITE""","""NOT HISPANIC OR LATINO""","""Pbo""","""Placebo""","""2014-01-02""","""2014-07-02""","""2014-01-01""","""CDISCPILOT01""","""DM""","""01-701-1015""","""01-701-1015""","""701""",63,"""YEARS""","""F""","""WHITE""","""NOT HISPANIC OR LATINO""","""PBO""","""Placebo""","""Pbo""","""Placebo""","""2014-01-02""","""2014-07-02""","""2014-01-01"""
"""01-701-1023""",701,64,"""F""","""WHITE""","""NOT HISPANIC OR LATINO""","""Xan_Lo""","""Xanomeline Low Dose""","""2014-01-03""","""2014-07-04""","""2014-01-02""","""CDISCPILOT01""","""DM""","""01-701-1023""","""01-701-1023""","""701""",64,"""YEARS""","""F""","""WHITE""","""NOT HISPANIC OR LATINO""","""XAN_LO""","""Xanomeline Low Dose""","""Xan_Lo""","""Xanomeline Low Dose""","""2014-01-03""","""2014-07-04""","""2014-01-02"""
"""01-701-1028""",701,71,"""M""","""WHITE""","""NOT HISPANIC OR LATINO""","""Xan_Hi""","""Xanomeline High Dose""","""2014-01-04""","""2014-07-05""","""2014-01-03""","""CDISCPILOT01""","""DM""","""01-701-1028""","""01-701-1028""","""701""",71,"""YEARS""","""M""","""WHITE""","""NOT HISPANIC OR LATINO""","""XAN_HI""","""Xanomeline High Dose""","""Xan_Hi""","""Xanomeline High Dose""","""2014-01-04""","""2014-07-05""","""2014-01-03"""


## Step 2b–2d — Synthesise EX, DS, AE

In the CDISC Pilot study these come from `{pharmaversesdtm}`. Here we reconstruct them from `dm_raw` and `ae_raw`.

In [4]:
# EX — Exposure (one record per subject)
ex = (
    dm_raw.with_columns(
        pl.lit("CDISCPILOT01").alias("STUDYID"),
        pl.lit("EX").alias("DOMAIN"),
        pl.col("patient_number").alias("USUBJID"),
        pl.col("arm_description").alias("EXTRT"),
        pl.when(pl.col("arm_code") == "Pbo").then(pl.lit(0.0))
          .when(pl.col("arm_code") == "Xan_Lo").then(pl.lit(54.0))
          .when(pl.col("arm_code") == "Xan_Hi").then(pl.lit(81.0))
          .otherwise(None).alias("EXDOSE"),
        pl.lit("mg").alias("EXDOSU"),
        pl.col("first_dose_date").alias("EXSTDTC"),
        pl.col("last_dose_date").alias("EXENDTC"),
        pl.lit(1).alias("EXSEQ"),
    )
    .select(["STUDYID", "DOMAIN", "USUBJID", "EXSEQ",
             "EXTRT", "EXDOSE", "EXDOSU", "EXSTDTC", "EXENDTC"])
)

# DS — Disposition
_ds_cols = ["STUDYID", "DOMAIN", "USUBJID", "DSSEQ",
            "DSCAT", "DSTERM", "DSDECOD", "DSSTDTC"]

ds = pl.concat([
    dm_raw.with_columns(
        pl.lit("CDISCPILOT01").alias("STUDYID"),
        pl.lit("DS").alias("DOMAIN"),
        pl.col("patient_number").alias("USUBJID"),
        pl.lit(1).alias("DSSEQ"),
        pl.lit("DISPOSITION EVENT").alias("DSCAT"),
        pl.lit("PROTOCOL COMPLETED").alias("DSTERM"),
        pl.lit("COMPLETED").alias("DSDECOD"),
        pl.col("last_dose_date").alias("DSSTDTC"),
    ).select(_ds_cols),
    dm_raw.with_columns(
        pl.lit("CDISCPILOT01").alias("STUDYID"),
        pl.lit("DS").alias("DOMAIN"),
        pl.col("patient_number").alias("USUBJID"),
        pl.lit(2).alias("DSSEQ"),
        pl.lit("PROTOCOL MILESTONE").alias("DSCAT"),
        pl.lit("RANDOMIZED").alias("DSTERM"),
        pl.lit("RANDOMIZED").alias("DSDECOD"),
        pl.col("first_dose_date").alias("DSSTDTC"),
    ).select(_ds_cols),
]).sort(["USUBJID", "DSSEQ"])

# AE — Adverse Events
ae = (
    ae_raw.with_columns(
        pl.lit("CDISCPILOT01").alias("STUDYID"),
        pl.lit("AE").alias("DOMAIN"),
        pl.col("patient_number").alias("USUBJID"),
        pl.col("ae_term").alias("AETERM"),
        pl.col("ae_start_date").alias("AESTDTC"),
        pl.col("ae_end_date").alias("AEENDTC"),
        pl.col("ae_severity").alias("AESEV"),
        pl.col("ae_serious").alias("AESER"),
        pl.col("ae_outcome").alias("AEOUT"),
        pl.col("ae_action_taken").alias("AEACN"),
        pl.col("ae_relationship").alias("AEREL"),
        pl.col("ae_term").cum_count().over("patient_number").alias("AESEQ"),
    )
    .select(["STUDYID", "DOMAIN", "USUBJID", "AESEQ",
             "AETERM", "AESTDTC", "AEENDTC", "AESEV",
             "AESER", "AEOUT", "AEACN", "AEREL"])
)

print(f"EX  : {ex.shape}")
print(f"DS  : {ds.shape}")
print(f"AE  : {ae.shape}")

EX  : (5, 9)
DS  : (10, 8)
AE  : (5, 12)


## Steps 3–5 — ADSL Base + Treatment Dates + Duration

| admiral | fleet |
|---------|----------|
| `mutate(TRT01P = ARM)` | `.rename("TRT01P", source="ARM")` |
| `derive_vars_merged(EX, mode="first")` | `.first_date("TRTSDT", ...)` |
| `derive_vars_merged(EX, mode="last")` | `.last_date("TRTEDT", ...)` |
| `derive_var_trtdurd()` | `.duration("TRTDURD", ...)` |

In [5]:
adsl = (
    dm.adsl
    .source("STUDYID")
    .source("USUBJID")
    .source("SUBJID")
    .source("SITEID")
    .source("AGE")
    .source("AGEU")
    .source("SEX")
    .source("RACE")
    .source("ETHNIC")
    .source("ARM")
    .source("ARMCD")
    .source("ACTARM")
    .source("ACTARMCD")
    .rename("TRT01P", source="ARM")
    .rename("TRT01A", source="ACTARM")
    .source("RFSTDTC")
    .source("RFENDTC")
    .source("RFICDTC")
    .to_frame()
    .adsl
    .first_date("TRTSDT", from_dataset=ex, date_col="EXSTDTC")
    .last_date( "TRTEDT", from_dataset=ex, date_col="EXENDTC")
    .duration(  "TRTDURD", start_date="TRTSDT", end_date="TRTEDT")
    .to_frame()
)

print(f"ADSL: {adsl.shape}")
adsl.select(["USUBJID", "TRT01P", "TRTSDT", "TRTEDT", "TRTDURD"]).head(5)

ADSL: (5, 33)


USUBJID,TRT01P,TRTSDT,TRTEDT,TRTDURD
str,str,str,str,i64
"""01-701-1015""","""Placebo""","""2014-01-02""","""2014-07-02""",182
"""01-701-1023""","""Xanomeline Low Dose""","""2014-01-03""","""2014-07-04""",183
"""01-701-1028""","""Xanomeline High Dose""","""2014-01-04""","""2014-07-05""",183
"""01-701-1034""","""Placebo""","""2014-01-05""","""2014-07-06""",183
"""01-701-1047""","""Xanomeline Low Dose""","""2014-01-06""","""2014-07-07""",183


## Step 6 — Disposition Variables (EOSDT, EOSSTT, DCSREAS, RANDDT)

In [6]:
ds_eosdt = (
    ds.filter(pl.col("DSCAT") == "DISPOSITION EVENT")
      .group_by("USUBJID")
      .agg(pl.col("DSSTDTC").max().alias("EOSDT"))
)

ds_eosstt = (
    ds.filter(pl.col("DSCAT") == "DISPOSITION EVENT")
      .with_columns(
          pl.when(pl.col("DSDECOD") == "COMPLETED").then(pl.lit("COMPLETED"))
            .when(pl.col("DSDECOD") == "SCREEN FAILURE").then(pl.lit(None))
            .otherwise(pl.lit("DISCONTINUED"))
            .alias("EOSSTT")
      )
      .select(["USUBJID", "EOSSTT"])
)

ds_dcsreas = (
    ds.filter(
        (pl.col("DSCAT") == "DISPOSITION EVENT")
        & pl.col("DSDECOD").is_not_null()
        & (~pl.col("DSDECOD").is_in(["COMPLETED", "SCREEN FAILURE"]))
    )
    .select(pl.col("USUBJID"),
            pl.col("DSDECOD").alias("DCSREAS"),
            pl.col("DSTERM").alias("DCSREASP"))
)

ds_randdt = (
    ds.filter(pl.col("DSDECOD") == "RANDOMIZED")
      .select(pl.col("USUBJID"), pl.col("DSSTDTC").alias("RANDDT"))
)

adsl = (
    adsl
    .join(ds_eosdt,   on="USUBJID", how="left")
    .join(ds_eosstt,  on="USUBJID", how="left")
    .join(ds_dcsreas, on="USUBJID", how="left")
    .join(ds_randdt,  on="USUBJID", how="left")
)

adsl["EOSSTT"].value_counts()

EOSSTT,count
str,u32
"""COMPLETED""",5


## Step 7 — Age Grouping (AGEGR1)

Admiral's `derive_vars_cat` → Fleet's `.condderivation()`.

In [7]:
adsl = (
    adsl.adsl
    .condderivation("AGEGR1",
                    conditions=[
                        {"condition": "pl.col('AGE') < 18",  "value": "<18"},
                        {"condition": "pl.col('AGE') <= 64", "value": "18-64"},
                        {"condition": "pl.col('AGE') > 64",  "value": ">64"},
                    ],
                    else_value="Missing")
    .to_frame()
)

adsl.group_by("AGEGR1").agg(pl.len().alias("n")).sort("AGEGR1")

AGEGR1,n
str,u32
"""18-64""",3
""">64""",2


## Step 8 — Population Flags (SAFFL, ITTFL)

In [8]:
subjects_exposed = ex.select("USUBJID").unique()

adsl = (
    adsl
    .with_columns(
        pl.col("USUBJID")
          .is_in(subjects_exposed["USUBJID"].to_list())
          .map_elements(lambda v: "Y" if v else None, return_dtype=pl.String)
          .alias("SAFFL")
    )
    .adsl
    .eligibility_flag("ITTFL", criteria='pl.col("ARMCD") != "SCRNFAIL"')
    .to_frame()
)

print("SAFFL:", adsl["SAFFL"].value_counts().to_dicts())
print("ITTFL:", adsl["ITTFL"].value_counts().to_dicts())

SAFFL: [{'SAFFL': 'Y', 'count': 5}]
ITTFL: [{'ITTFL': 'Y', 'count': 5}]


## Step 9 — Last Date Known Alive (LSTALVDT)

Admiral's `derive_vars_extreme_event` — Fleet uses `pl.max_horizontal` across AE dates and TRTEDT.

In [9]:
ae_last_end = (
    ae.filter(pl.col("AEENDTC").is_not_null())
      .group_by("USUBJID")
      .agg(pl.col("AEENDTC").max().alias("_ae_end"))
)

ae_last_start = (
    ae.filter(pl.col("AESTDTC").is_not_null())
      .group_by("USUBJID")
      .agg(pl.col("AESTDTC").max().alias("_ae_start"))
)

adsl = (
    adsl
    .join(ae_last_end,   on="USUBJID", how="left")
    .join(ae_last_start, on="USUBJID", how="left")
    .with_columns(
        pl.max_horizontal(
            pl.col("_ae_end"), pl.col("_ae_start"), pl.col("TRTEDT")
        ).alias("LSTALVDT")
    )
    .drop(["_ae_end", "_ae_start"])
)

print(f"LSTALVDT range: {adsl['LSTALVDT'].min()} → {adsl['LSTALVDT'].max()}")

LSTALVDT range: 2014-07-02 → 2014-07-07


## Step 10 — AE Summary Counts (NUMAE, NUMSAE)

In [10]:
adsl = (
    adsl.adsl
    .count_events("NUMAE",  from_dataset=ae)
    .count_events("NUMSAE", from_dataset=ae, where='pl.col("AESER") == "Y"')
    .to_frame()
)

print(f"Subjects with ≥1 AE  : {(adsl['NUMAE']  > 0).sum()}")
print(f"Subjects with ≥1 SAE : {(adsl['NUMSAE'] > 0).sum()}")

Subjects with ≥1 AE  : 4
Subjects with ≥1 SAE : 1


## Step 11 — Final Column Selection

In [11]:
FINAL_COLS = [
    "STUDYID", "USUBJID", "SUBJID", "SITEID",
    "AGE", "AGEU", "AGEGR1", "SEX", "RACE", "ETHNIC",
    "ARM", "ARMCD", "ACTARM", "ACTARMCD",
    "TRT01P", "TRT01A",
    "RFSTDTC", "RFENDTC", "RFICDTC",
    "RANDDT", "TRTSDT", "TRTEDT", "TRTDURD",
    "EOSDT", "EOSSTT", "DCSREAS", "DCSREASP",
    "LSTALVDT",
    "SAFFL", "ITTFL",
    "NUMAE", "NUMSAE",
]

adsl_final = adsl.select(FINAL_COLS)
print(f"Final ADSL: {adsl_final.shape[0]} subjects × {adsl_final.shape[1]} variables")
adsl_final.head(5)

Final ADSL: 5 subjects × 32 variables


STUDYID,USUBJID,SUBJID,SITEID,AGE,AGEU,AGEGR1,SEX,RACE,ETHNIC,ARM,ARMCD,ACTARM,ACTARMCD,TRT01P,TRT01A,RFSTDTC,RFENDTC,RFICDTC,RANDDT,TRTSDT,TRTEDT,TRTDURD,EOSDT,EOSSTT,DCSREAS,DCSREASP,LSTALVDT,SAFFL,ITTFL,NUMAE,NUMSAE
str,str,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,u32,u32
"""CDISCPILOT01""","""01-701-1015""","""01-701-1015""","""701""",63,"""YEARS""","""18-64""","""F""","""WHITE""","""NOT HISPANIC OR LATINO""","""Placebo""","""PBO""","""Placebo""","""Pbo""","""Placebo""","""Placebo""","""2014-01-02""","""2014-07-02""","""2014-01-01""","""2014-01-02""","""2014-01-02""","""2014-07-02""",182,"""2014-07-02""","""COMPLETED""",null,null,"""2014-07-02""","""Y""","""Y""",2,0
"""CDISCPILOT01""","""01-701-1023""","""01-701-1023""","""701""",64,"""YEARS""","""18-64""","""F""","""WHITE""","""NOT HISPANIC OR LATINO""","""Xanomeline Low Dose""","""XAN_LO""","""Xanomeline Low Dose""","""Xan_Lo""","""Xanomeline Low Dose""","""Xanomeline Low Dose""","""2014-01-03""","""2014-07-04""","""2014-01-02""","""2014-01-03""","""2014-01-03""","""2014-07-04""",183,"""2014-07-04""","""COMPLETED""",null,null,"""2014-07-04""","""Y""","""Y""",1,0
"""CDISCPILOT01""","""01-701-1028""","""01-701-1028""","""701""",71,"""YEARS""",""">64""","""M""","""WHITE""","""NOT HISPANIC OR LATINO""","""Xanomeline High Dose""","""XAN_HI""","""Xanomeline High Dose""","""Xan_Hi""","""Xanomeline High Dose""","""Xanomeline High Dose""","""2014-01-04""","""2014-07-05""","""2014-01-03""","""2014-01-04""","""2014-01-04""","""2014-07-05""",183,"""2014-07-05""","""COMPLETED""",null,null,"""2014-07-05""","""Y""","""Y""",1,0
"""CDISCPILOT01""","""01-701-1034""","""01-701-1034""","""701""",74,"""YEARS""",""">64""","""M""","""BLACK OR AFRICAN AMERICAN""","""NOT HISPANIC OR LATINO""","""Placebo""","""PBO""","""Placebo""","""Pbo""","""Placebo""","""Placebo""","""2014-01-05""","""2014-07-06""","""2014-01-04""","""2014-01-05""","""2014-01-05""","""2014-07-06""",183,"""2014-07-06""","""COMPLETED""",null,null,"""2014-07-06""","""Y""","""Y""",1,1
"""CDISCPILOT01""","""01-701-1047""","""01-701-1047""","""701""",56,"""YEARS""","""18-64""","""F""","""WHITE""","""NOT HISPANIC OR LATINO""","""Xanomeline Low Dose""","""XAN_LO""","""Xanomeline Low Dose""","""Xan_Lo""","""Xanomeline Low Dose""","""Xanomeline Low Dose""","""2014-01-06""","""2014-07-07""","""2014-01-05""","""2014-01-06""","""2014-01-06""","""2014-07-07""",183,"""2014-07-07""","""COMPLETED""",null,null,"""2014-07-07""","""Y""","""Y""",0,0


## Step 12 — Validation Summary

In [12]:
summary = {
    "Total subjects": adsl_final.shape[0],
    "Treatment arms": sorted(adsl_final["TRT01P"].unique().to_list()),
    "Age range": f"{adsl_final['AGE'].min()} – {adsl_final['AGE'].max()} years",
    "SAFFL = Y": int((adsl_final["SAFFL"] == "Y").sum()),
    "ITTFL = Y": int((adsl_final["ITTFL"] == "Y").sum()),
    "TRTDURD range": f"{adsl_final['TRTDURD'].min()} – {adsl_final['TRTDURD'].max()} days",
    "Subjects ≥1 AE": int((adsl_final["NUMAE"] > 0).sum()),
    "Subjects ≥1 SAE": int((adsl_final["NUMSAE"] > 0).sum()),
}

for k, v in summary.items():
    print(f"  {k:<25}: {v}")

  Total subjects           : 5
  Treatment arms           : ['Placebo', 'Xanomeline High Dose', 'Xanomeline Low Dose']
  Age range                : 56 – 74 years
  SAFFL = Y                : 5
  ITTFL = Y                : 5
  TRTDURD range            : 182 – 183 days
  Subjects ≥1 AE           : 4
  Subjects ≥1 SAE          : 1


## Save Output

In [13]:
OUT.mkdir(parents=True, exist_ok=True)
adsl_final.write_parquet(OUT / "adsl_admiral.parquet")
adsl_final.write_csv(OUT / "adsl_admiral.csv")
print(f"Saved to {OUT}/adsl_admiral.{{parquet,csv}}")

Saved to ../../outputs/cdiscpilot01/adsl_admiral.{parquet,csv}
